# Clase 1.1: Ingeniería de Prompts (Prompt Engineering)
**ECI 2026 - Agentes de Inteligencia Artificial - UBA - Julio 2026**

¡Bienvenidos al taller práctico de Ingeniería de Prompts! Como vimos en la teoría, el prompting es la forma mediante la cual los humanos nos comunicamos y guiamos a los modelos de lenguaje.

En este notebook recorreremos desde los conceptos iniciales y básicos de comunicación con LLMs hasta técnicas avanzadas que sientan las bases para la construcción de **Agentes de IA**. Usaremos modelos a través de Hugging Face y Google AI Studio.


In [ ]:
# Instalamos las librerías necesarias
!pip install -q huggingface_hub google-generativeai pypdf2 groq requests pillow


In [ ]:
import os
import requests
from PIL import Image
from io import BytesIO
from huggingface_hub import InferenceClient
import google.genai as genai
from google.genai import types
from google.colab import userdata

# Instrucciones:
# 1. Token de Hugging Face en: https://huggingface.co/settings/tokens
# 2. API Key de Gemini en: https://aistudio.google.com/app/apikey
# 3. Guarden las claves en 'Secrets' de Colab con los nombres HF_TOKEN y GEMINI_API_KEY.

hf_token = userdata.get('HF_TOKEN')
gemini_key = userdata.get('GEMINI_API_KEY')

# Inicializamos los clientes
cliente_hf = InferenceClient(token=hf_token)
client = genai.Client(api_key=gemini_key)

MODELO_GEMINI = 'gemini-2.5-flash'
print("Entorno configurado y clientes inicializados.")


## PARTE 1: Modelos Base, Formatos y Grounding
En esta primera etapa usaremos la función `generate_content` con modelos de lenguaje para ver sus capacidades iniciales.

### Ejercicio 1 - Ambigüedad vs. Claridad
**Recomendación:** Comunicate de forma clara y precisa. Elimina asunciones.

**Tu tarea:** El prompt abajo es ambiguo. Iterá y cambialo para pedir algo hiperespecífico (ej. un resumen de 3 puntos sobre algoritmos para un niño de 10 años).

In [ ]:


prompt_generico = "El colegio de abogados de la Ciudad de Buenos Aires a que hora abre?"


print("\n--- Consultando API de Google Gemini ---")


try:
    # Generar el contenido
    respuesta_gemini = client.models.generate_content(contents=prompt_generico, model=MODELO_GEMINI)
    print("\nRespuesta Gemini (API):\n", respuesta_gemini.text)

except Exception as e:
    print(f"Ocurrió un error al llamar a la API: {e}")

### Ejercicio 2 - Formatos y Restricciones
Las LLMs tienen un límite de tokens de entrada y salida.

**Tu tarea:** Escribí un prompt resolviendo un problema de tu elección. Obligá al modelo a responder **exactamente en un formato JSON** y con un máximo de 50 palabras.

In [ ]:
prompt_formato = """
Escribe un diccionario en Python con 3 lenguajes de programación y su año de creación.
Devuelve únicamente el JSON en formato JSON estricto:
"""

respuesta_formato = client.models.generate_content(contents=prompt_formato, model=MODELO_GEMINI)
print("Respuesta JSON:\n", respuesta_formato.text)

### Ejercicio 3 - Grounding con Archivos (PDF)
Con técnicas de grounding podemos pasar a la LLM otro tipo de entradas como archivos que se usan como contexto para responder.

**Tu tarea:** Subí un PDF corto a Colab. Usaremos Gemini para extraer información específica.

In [ ]:
from google.colab import files
import os

# 1. Abre el selector de archivos
uploaded = files.upload()

# 2. Obtener el nombre del archivo subido
for filename in uploaded.keys():
    print(f'Archivo "{filename}" subido con éxito.')

    # Guardar la ruta para usarla después
    pdf_path = os.path.join(os.getcwd(), filename)
    print(f'Ruta del PDF: {pdf_path}')

In [ ]:
!ls


In [ ]:
import PyPDF2

nombre_archivo = 'NIPS-2015-hidden-technical-debt-in-machine-learning-systems-Paper.pdf'

texto_pdf = ""
try:
    with open(nombre_archivo, 'rb') as archivo:
        lector = PyPDF2.PdfReader(archivo)
        for pagina in lector.pages:
            texto_pdf += pagina.extract_text()

    prompt_grounding = f"""
    Basado EXCLUSIVAMENTE en el siguiente texto, responde la pregunta.

    Texto: {texto_pdf[:2000]}

    Pregunta 1: ¿Cuál es el título del documento?
    Pregunta 2: ¿Cuáles son las 3 ideas principales del documento?
    """

    respuesta_gemini = client.models.generate_content(contents=prompt_grounding, model=MODELO_GEMINI)
    print("Respuesta con Grounding (Gemini):\n", respuesta_gemini.text)
except FileNotFoundError:
    print("Por favor, recuerda subir un PDF al entorno.")

In [ ]:
texto_pdf[:2000]

---
## PARTE 2: Evolución a Modelos Instruct y Formato Chat
Si intentamos usar `text_generation` con modelos modernos optimizados para instrucciones (como `Llama-3-8B-Instruct`), nos dará error. Estos modelos requieren la función `chat_completion` y un formato estructurado de mensajes (`role` y `content`).

### Ejercicio 4 - Etapas de Reflexión (Chain-of-Thought)
**Tu tarea:** Vamos a plantearle un problema engañoso al modelo. Usaremos un prompt estructurado en etapas, forzando al modelo a "pensar en voz alta" paso a paso antes de dar la conclusión final.

In [ ]:
problema_logico = "Un guante de arquero y una pelota cuestan $1.10 en total. El guante cuesta $1.00 más que la pelota. ¿Cuánto cuesta la pelota?"

prompt_reflexion = f"""
Resuelve este problema siguiendo estrictamente estas etapas de reflexión:
Etapa 1 - Análisis: Desglosa los datos.
Etapa 2 - Ecuación: Escribe la ecuación matemática.
Etapa 3 - Resolución: Resuelve paso a paso.
Etapa 4 - Conclusión: Declara el precio exacto de la pelota.

Problema: {problema_logico}
"""

# Armamos la estructura de chat que exige el modelo
mensajes_cot = [{"role": "user", "content": prompt_reflexion}]

respuesta_reflexion = cliente_hf.chat_completion(
    messages=mensajes_cot,
    model="meta-llama/Llama-3.1-8B-Instruct",
    max_tokens=600 # Nota: En chat_completion se usa max_tokens
)

print("--- RESPUESTA CON REFLEXIÓN ---\n")
print(respuesta_reflexion.choices[0].message.content)

### Ejercicio 5 - Instruction Prompting con Roles del Sistema
La ventaja del formato Chat es que podemos separar la personalidad del modelo de la instrucción del usuario usando el rol `system`.

**Tu tarea:** Ejecuta el código. Luego, modifica el System Prompt para asignarle un rol distinto (por ejemplo, un chef explicando una receta a un novato) y observa cómo cambia radicalmente la salida.

In [ ]:
# Definimos el comportamiento general en el rol 'system'
system_prompt = """
Eres un profesor universitario experto en Ciencias de la Computación.
Tu tono debe ser didáctico, empático y alentador.
Responde siempre en menos de 100 palabras usando analogías cotidianas.
"""

# Definimos el requerimiento específico en el rol 'user'
user_prompt = "Explica el concepto de 'Machine Learning' a estudiantes de primer año."

mensajes_instruccion = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

respuesta_instruccion = cliente_hf.chat_completion(
    messages=mensajes_instruccion,
    model="meta-llama/Llama-3.1-8B-Instruct",
    max_tokens=150
)

print("--- RESPUESTA CON ROLES DEL SISTEMA ---\n")
print(respuesta_instruccion.choices[0].message.content)

---
## PARTE 3: Técnicas Intermedias: Zero-shot, Few-shot y Multimodalidad
Las LLMs son excelentes adaptándose al contexto.
* **Zero-shot:** Le pedimos una tarea sin darle ejemplos previos. Útil para tareas genéricas.
* **Few-shot:** Le damos 2 o 3 ejemplos resolviendo el problema para que identifique el patrón y el formato exacto que necesitamos.

### Ejercicio 6 - Zero-shot vs. Few-shot Prompting

In [ ]:
print("--- ZERO-SHOT ---")
prompt_zero = "Clasifica el sentimiento de esta reseña: 'El algoritmo es eficiente, pero la interfaz es incomprensible'."
resp_zero = client.models.generate_content(contents=prompt_zero, model=MODELO_GEMINI)
print(resp_zero.text, "\n")

print("--- FEW-SHOT ---")
prompt_few = """
Clasifica el sentimiento de los textos técnicos. Devuelve SOLO una palabra: POSITIVO, NEGATIVO o NEUTRAL.

Texto: El tiempo de compilación bajó un 40% con este refactor.
Sentimiento: POSITIVO

Texto: Tuvimos un error de desbordamiento de memoria en el servidor en la nube.
Sentimiento: NEGATIVO

Texto: La librería está escrita en Python y C++.
Sentimiento: NEUTRAL

Texto: El algoritmo es eficiente, pero la interfaz es incomprensible.
Sentimiento:
"""
resp_few = client.models.generate_content(contents=prompt_few, model=MODELO_GEMINI)
print(resp_few.text)

### Ejercicio 7 - Few-Shot Multimodal (Imágenes)
Los modelos modernos (como Gemini 2.5) son nativamente multimodales. Podemos aplicar la misma lógica del *Few-Shot* pasándole una secuencia de imágenes y textos para que aprenda un patrón visual en el momento (In-Context Learning).

En este ejemplo, le enseñaremos a clasificar imágenes con un formato específico.

In [ ]:
# Función auxiliar para cargar imágenes desde internet con encabezados de seguridad
def cargar_imagen(url):
    # Disfrazamos la petición como si fuera un navegador web normal
    headers = {
        'User-Agent': 'Chrome/120.0.0.0'
    }
    respuesta = requests.get(url, headers=headers)

    # Validamos que la descarga fue exitosa antes de procesarla
    respuesta.raise_for_status()

    return Image.open(BytesIO(respuesta.content))

# 1. Cargamos algunas imágenes de prueba (Manzana, Naranja y una Banana para la prueba final)
url_manzana = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Red_Apple.jpg/120px-Red_Apple.jpg"
url_naranja = "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c4/Orange-Fruit-Pieces.jpg/120px-Orange-Fruit-Pieces.jpg"
url_banana = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8a/Banana-Single.jpg/120px-Banana-Single.jpg"

img_ejemplo_1 = cargar_imagen(url_manzana)
img_ejemplo_2 = cargar_imagen(url_naranja)
img_pregunta = cargar_imagen(url_banana)

# 2. Construimos el prompt Few-Shot intercalando texto e imágenes en una lista
prompt_multimodal = [
    "Eres un clasificador botánico estricto. Analiza las imágenes y responde con el formato: 'Clase: [Nombre]'",
    img_ejemplo_1,
    "Clase: Manzana",
    img_ejemplo_2,
    "Clase: Naranja",
    img_pregunta,
    "Clase:" # Dejamos la última parte abierta para que el modelo complete
]

print("Consultando a Gemini con Few-Shot Multimodal...")
resp_vision = client.models.generate_content(contents=prompt_multimodal, model=MODELO_GEMINI)
print("\nRespuesta del modelo:", resp_vision.text)

---
## PARTE 4: Razonamiento Avanzado y Agentes
En lugar de pedir la respuesta final de inmediato, obligamos al modelo a "pensar en voz alta". Esto reduce dramáticamente las alucinaciones en problemas de lógica, matemáticas y programación.

### Ejercicio 8 - Chain-of-Thought (CoT)

In [ ]:
prompt_cot = """
Pregunta: En un clúster de servidores, tengo 10 GPUs. Asigno la mitad a un entrenamiento profundo. Luego, adquiero 4 GPUs más y le presto 2 a un colega. ¿Cuántas GPUs tengo disponibles para uso inmediato?

Vamos a pensar paso a paso.
Respuesta:
"""
resp_cot = client.models.generate_content(contents=prompt_cot, model=MODELO_GEMINI)
print(resp_cot.text)

### Ejercicio 9 - El framework ReAct (Reasoning and Acting)
Esta es la piedra angular de los **Agentes de IA**. ReAct combina el razonamiento (CoT) con la capacidad de tomar acciones e interactuar con el entorno (herramientas). El agente entra en un bucle de Pensamiento -> Acción -> Observación.

In [ ]:
# Simularemos el patrón de razonamiento de un Agente sin ejecutar herramientas reales (por ahora).
prompt_react = """
Eres un Agente autónomo. Resuelve la pregunta simulando el uso de herramientas.
Herramientas disponibles: [BuscadorWeb, Calculadora]

Utiliza estrictamente este formato iterativo:
Pensamiento: [Qué debo hacer]
Acción: [Nombre de herramienta] - [Input de la herramienta]
Observación: [Lo que imaginas que devuelve la herramienta]
... (repetir hasta tener la solución)
Respuesta Final: [Conclusión]

Pregunta: ¿Cuánto es el doble de la cantidad de años desde que se fundó la Universidad de Buenos Aires (UBA)?
"""
resp_react = client.models.generate_content(contents=prompt_react, model=MODELO_GEMINI)
print(resp_react.text)